In [138]:
import re
import numpy as np
import pandas as pd
from collections import Counter
from nltk.corpus import stopwords
import nltk

In [139]:
## Leer los datos
df_tortugas = (pd.read_csv('../demo_datasets/demo_tortugas.csv', sep=';')
                #.head()
                .dropna(subset=['observa']))
df_tortugas.head()

,id,ficha,especie,nombre.especie,fecha_orig,fecha,anio,mes,estacion,lugar_orig,...,fmt_lugar,muni,codmun,causa_orig,causa,muerte,observa,lesion,cuerpo,estado
1,2,1868 - 1977-1990,Caretta caretta,Tortuga Boba,14/11/1989,14/11/1989,1989,Noviembre,Otoño,NaN,...,NaN,Icod de los Vinos,38022,Cautividad,Otros,No,3 años en cautividad en agua dulce.,NaN,NaN,NaN
2,3,9537 - 1998-2010,Caretta caretta,Tortuga Boba,02/12/2010,02/12/2010,2010,Diciembre,Invierno,CANDELARIA - CANDELARIA,...,"Candelaria, Santa Cruz de Tenerife, Islas Cana...",Candelaria,38011,Enfermedad,Enfermedad,No,"Le falta la aleta delantera dcha, caparazón y ...",Herida,Varias partes,NaN
3,4,9521 - 1998-2010,Caretta caretta,Tortuga Boba,15/11/2010,15/11/2010,2010,Noviembre,Otoño,Puerto Colón,...,"Puerto, Tazacorte, Santa Cruz de Tenerife, Isl...",Adeje,38001,Artes de pesca,Artes de pesca,Si,"Corte en el cuello por enmallamiento, flaca, d...",Varias lesiones,Cuello,NaN
4,5,9436 - 1998-2010,Caretta caretta,Tortuga Boba,29/09/2010,29/09/2010,2010,Septiembre,Otoño,EL PORIS - ARICO,...,"Carretera al Porís, 38588, Arico, Santa Cruz d...",Arico,38005,Artes de pesca,Artes de pesca,No,"aleta delantera dcha necrosada, jaime 15/10",NaN,Aleta,Necrosada
5,6,9433 - 1998-2010,Caretta caretta,Tortuga Boba,28/09/2010,28/09/2010,2010,Septiembre,Otoño,las teresitas,...,Playa de las Teresitas,Santa Cruz de Tenerife,38038,Artes de pesca,Artes de pesca,No,aleta delantera y trasera dchas con cortes por...,Varias lesiones,Aleta,NaN


In [119]:
##Descargar stopwords y seleccionar las de español
nltk.download('stopwords')
stop_words = set(stopwords.words('spanish'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jcge9\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [122]:
def extract_words(text):
    words = re.findall(r'\b[a-záéíóúñ]+\b', text.lower())
    # ❗ quitamos stopwords
    return [w for w in words if w not in stop_words]

In [127]:
##Construcción del vocabulario
words = []

for t in df_tortugas['observa']:
    words.extend(extract_words(t))

word_freq = Counter(words)

vocab = [w for w, f in word_freq.items() if f >= 1]

In [128]:
def get_embedding(text):
    return client.embeddings.create(
        input=text,
        model="text-embedding-3-small"
    ).data[0].embedding

word_emb = {w: get_embedding(w) for w in vocab}

In [130]:
## Buscar sinónimos de una palabra 
from sklearn.metrics.pairwise import cosine_similarity

def find_synonyms(word, word_emb, top_k=10):
    if word not in word_emb:
        return f"'{word}' no está en el vocabulario"

    target = np.array(word_emb[word]).reshape(1, -1)
    
    sims = []
    for w, emb in word_emb.items():
        if w == word:
            continue
        sim = cosine_similarity(
            target,
            np.array(emb).reshape(1, -1)
        )[0][0]
        sims.append((w, sim))
    
    sims.sort(key=lambda x: x[1], reverse=True)
    return sims[:top_k]

In [137]:
find_synonyms("anzuelo", word_emb, top_k=15)

[('pescador', np.float64(0.4523883376026811)),
 ('atada', np.float64(0.44262894313954876)),
 ('aleta', np.float64(0.43592513111436726)),
 ('ano', np.float64(0.43416709162408584)),
 ('caparazon', np.float64(0.4253071198599472)),
 ('delantera', np.float64(0.42509191607723684)),
 ('alta', np.float64(0.41577398998835)),
 ('enrredada', np.float64(0.4126207485059057)),
 ('suelta', np.float64(0.40593739676755114)),
 ('trasera', np.float64(0.4028479734229714)),
 ('cuello', np.float64(0.4021611741580977)),
 ('pesca', np.float64(0.4001492183275682)),
 ('enredad', np.float64(0.40008784738876757)),
 ('arrastraba', np.float64(0.39670092037706994)),
 ('enmalladas', np.float64(0.39542269374867706))]